# P 本地 G3 的 SLA 逐请求对照
使用同一 trace 的请求配对比较两次全量回放。

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import zip_longest
from pathlib import Path
import csv, json
import pandas as pd
import matplotlib.pyplot as plt
import yaml

project = Path.home()/'project'
out = project/'aisimulate/experiments/disk/analysis/p_g3_local/assets'
out.mkdir(parents=True,exist_ok=True)
base = project/'results/glm52-1p1d-g2/20261105T094503.158905Z/p_g2_shared/20261105T094520.729529Z'
candidate = project/'results/glm52-1p1d-p-g3-local/20261105T122917.107018Z'
old,new = [json.loads((p/'run.json').read_text()) for p in (base,candidate)]
for key in ('selected_trace_sha256','selected_requests','speedup','warmup_seconds','cache_initial_state'):
    assert old[key] == new[key], key
assert old['speedup']==1 and all(x['status']=='completed' for x in (old,new))
sla, candidate_sla = [yaml.safe_load((p/'resolved.yaml').read_text())['evaluation']['sla'] for p in (base,candidate)]
assert sla == candidate_sla
with (Path(old['bundle'])/'source_requests.jsonl').open() as f:
    origin = json.loads(next(f))['start_ms']

def status(row):
    if row['terminal_status']!='completed' or row['output_length']!=row['requested_output_length']:
        return 'incomplete'
    flags = []
    if row['ttft_ms'] > sla['ttft_ms']:
        flags.append('TTFT')
    if row['output_length'] > 1 and row['itl_ms'] > sla['itl_ms']:
        flags.append('ITL')
    return '+'.join(flags) or 'pass'

def utc(ms):
    return datetime.fromtimestamp(ms/1000,timezone.utc).isoformat()

counts, reasons = Counter(), Counter()
lengths, windows = defaultdict(Counter), defaultdict(Counter)
flips = []
labels = {(True,True):'both_pass',(False,False):'both_fail',(False,True):'improved',(True,False):'regressed'}
with (base/'prediction/requests.jsonl').open() as a, (candidate/'prediction/requests.jsonl').open() as b:
    for left,right in zip_longest(a,b):
        assert left is not None and right is not None
        g2,g3 = json.loads(left),json.loads(right)
        assert all(g2[k]==g3[k] for k in ('session_id','arrival_time_ms','input_length','requested_output_length'))
        before,after = status(g2),status(g3)
        change = labels[before=='pass',after=='pass']
        counts[change]+=1
        n = g2['input_length']
        bucket = ['<32K','32–64K','64–96K','96–128K'][min(n//32768,3)]
        lengths[bucket][change]+=1
        arrival = origin+g2['arrival_time_ms']
        windows[int(arrival//300000)*300000][change]+=1
        if change in ('improved','regressed'):
            reasons[change,before if change=='improved' else after]+=1
            row = dict(session_id=g2['session_id'],arrival_utc=utc(arrival),input_tokens=n,
                       output_tokens=g2['requested_output_length'],change=change,g2_status=before,g3_status=after)
            for label,rec in [('g2',g2),('g3',g3)]:
                for key in ('ttft_ms','itl_ms','first_admission_host_reused_input_tokens'):
                    row[f'{label}_{key}']=rec.get(key)
                row[f'{label}_decode_readmissions']=sum(x['pool']=='decode' and x['is_readmission'] for x in rec['admission_history'])
            flips.append(row)
assert sum(counts.values())==old['selected_requests']
# Reconcile per-request labels against native goodput without loading the large per-request array.
import ijson
for run, passing in [(base, counts['both_pass']+counts['regressed']),
                     (candidate, counts['both_pass']+counts['improved'])]:
    with (run/'prediction/prediction.json').open('rb') as f:
        assert next(ijson.items(f,'goodput_completed_requests')) == passing

def bins(groups, name):
    rows = []
    for key,c in groups:
        row = {name:key, **{v:c[v] for v in labels.values()}}
        row['requests'] = sum(c.values())
        row['net_gain'] = c['improved']-c['regressed']
        rows.append(row)
    return pd.DataFrame(rows)

sizes = bins(sorted(lengths.items(),key=lambda x:['<32K','32–64K','64–96K','96–128K'].index(x[0])),'input_length')
timeline = bins([(utc(k),v) for k,v in sorted(windows.items())],'start_utc')
sizes.to_csv(out/'by_input_length.csv',index=False)
timeline.to_csv(out/'by_arrival_5min.csv',index=False)
frame = pd.DataFrame(flips)
frame.to_csv(out/'sla_flips.csv',index=False)
profiles = {}
for direction,group in frame.groupby('change'):
    profiles[direction] = {
        'host_reuse_increased':int((group.g3_first_admission_host_reused_input_tokens>group.g2_first_admission_host_reused_input_tokens).sum()),
        'g2_decode_readmitted':int((group.g2_decode_readmissions>0).sum()),
        'g3_decode_readmitted':int((group.g3_decode_readmissions>0).sum())}
result = dict(baseline=str(base),candidate=str(candidate),trace_sha256=old['selected_trace_sha256'],
              sla=sla,counts=dict(counts),net_gain=counts['improved']-counts['regressed'],profiles=profiles,
              failure_reasons=[dict(change=k[0],reason=k[1],requests=v) for k,v in sorted(reasons.items())])
(out/'summary.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps(result,indent=2))
print(sizes.to_string(index=False))
print(timeline.nlargest(3,'net_gain').to_string(index=False))
print(timeline.nsmallest(3,'net_gain').to_string(index=False))
ax = sizes.set_index('input_length')[['improved','regressed']].plot.bar(figsize=(8,4),rot=0,color=['#248c7e','#c57250'])
ax.set(xlabel='Input tokens',ylabel='Requests',title='SLA changes after enabling local G3')
ax.figure.tight_layout()
ax.figure.savefig(out/'sla_by_input_length.png',dpi=160)
plt.show()
